# Experiment: TravelPlanner Sprint 6 Benchmark Runner

Objective:
- Exécuter un benchmark TravelPlanner sur plusieurs queries en local.
- Mesurer les métriques Sprint 6 (`final_pass_rate`, `hard_constraint_macro`, `commonsense_macro`).
- Vérifier la cible OC3: `final_pass_rate >= 0.322` (SwarmAgentic 32.2%).


In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean
from typing import Any

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'main.py').exists():
    raise RuntimeError('Run this notebook from the repository root.')

ROOT


## Plan

- Configurer le benchmark (subset de queries, ticks, agents).
- Préparer les données (`scripts/setup_travelplanner.py`).
- Lancer les runs via `main.py --adapter travelplanner`.
- Agréger les résultats + exporter un JSON sous `metrics/output/sprint6_travelplanner/`.


In [ ]:
# Benchmark configuration
DATA_DIR = Path('/tmp/travelplanner_db_check')
QUERY_INDICES = list(range(10))  # ex: [0, 1, 2] pour un pilote rapide
MAX_TICKS = 30
AGENTS = 3
SEED = 42
KEEP_SESSION = False
DATA_FORCE_DOWNLOAD = False
TARGET_SWARMAGENTIC = 0.322

ENV = os.environ.copy()

{
    'data_dir': str(DATA_DIR),
    'query_count': len(QUERY_INDICES),
    'max_ticks': MAX_TICKS,
    'agents': AGENTS,
    'seed': SEED,
    'target_final_pass_rate': TARGET_SWARMAGENTIC,
}


In [ ]:
def run_cmd(cmd: list[str], *, cwd: Path = ROOT, env: dict[str, str] | None = None) -> subprocess.CompletedProcess:
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if proc.returncode != 0:
        print('--- STDOUT (tail) ---')
        print(proc.stdout[-4000:])
        print('--- STDERR (tail) ---')
        print(proc.stderr[-4000:])
        raise RuntimeError(f'Command failed (exit={proc.returncode})')
    return proc

setup_cmd = [
    'uv',
    'run',
    'python',
    'scripts/setup_travelplanner.py',
    '--output-dir',
    str(DATA_DIR),
]
if DATA_FORCE_DOWNLOAD:
    setup_cmd.append('--force')

setup_proc = run_cmd(setup_cmd, env=ENV)
print(setup_proc.stdout.strip().splitlines()[-3:])


In [ ]:
JSON_DECODER = json.JSONDecoder()

def extract_summary_json(stdout: str) -> dict[str, Any]:
    starts = [m.start() for m in re.finditer(r'\{', stdout)]
    for start in reversed(starts):
        snippet = stdout[start:]
        try:
            payload, end = JSON_DECODER.raw_decode(snippet)
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict) and 'adapter' in payload and 'evaluation' in payload:
            return payload
    raise ValueError('Could not parse final JSON summary from CLI output.')

def run_query(query_idx: int) -> dict[str, Any]:
    cmd = [
        'uv',
        'run',
        'python',
        'main.py',
        '--adapter',
        'travelplanner',
        '--objective',
        f'Query {query_idx}',
        '--data-dir',
        str(DATA_DIR),
        '--max-ticks',
        str(MAX_TICKS),
        '--agents',
        str(AGENTS),
        '--seed',
        str(SEED + query_idx),
    ]
    if KEEP_SESSION:
        cmd.append('--keep-session')

    proc = run_cmd(cmd, env=ENV)
    summary = extract_summary_json(proc.stdout)
    summary['query_idx'] = query_idx
    return summary


In [ ]:
# Dry run rapide (3 queries max)
dry_indices = QUERY_INDICES[: min(3, len(QUERY_INDICES))]
dry_results = [run_query(i) for i in dry_indices]

pd.DataFrame([
    {
        'query_idx': r['query_idx'],
        'stop_reason': r.get('stop_reason'),
        'ticks': r.get('total_ticks'),
        'final_pass_rate': r.get('evaluation', {}).get('final_pass_rate'),
        'hard_constraint_macro': r.get('evaluation', {}).get('hard_constraint_macro'),
        'commonsense_macro': r.get('evaluation', {}).get('commonsense_macro'),
        'tokens': r.get('tokens_used'),
        'cost_usd': r.get('cost_used'),
    }
    for r in dry_results
])


In [ ]:
# Benchmark complet sur QUERY_INDICES
results_by_idx = {r['query_idx']: r for r in dry_results}
for idx in QUERY_INDICES:
    if idx in results_by_idx:
        continue
    print(f'\n=== Query {idx} ===')
    results_by_idx[idx] = run_query(idx)

results = [results_by_idx[idx] for idx in QUERY_INDICES if idx in results_by_idx]
len(results)


In [ ]:
def metric_avg(items: list[dict[str, Any]], name: str) -> float:
    values = [float(item.get('evaluation', {}).get(name, 0.0)) for item in items]
    return mean(values) if values else 0.0

aggregate = {
    'n_queries': len(results),
    'avg_final_pass_rate': metric_avg(results, 'final_pass_rate'),
    'avg_hard_constraint_macro': metric_avg(results, 'hard_constraint_macro'),
    'avg_commonsense_macro': metric_avg(results, 'commonsense_macro'),
    'avg_delivery_rate': metric_avg(results, 'delivery_rate'),
    'avg_tokens_used': mean(float(r.get('tokens_used', 0.0)) for r in results) if results else 0.0,
    'avg_cost_usd': mean(float(r.get('cost_used', 0.0)) for r in results) if results else 0.0,
    'target_swarmagentic_final_pass_rate': TARGET_SWARMAGENTIC,
}
aggregate['beats_swarmagentic'] = aggregate['avg_final_pass_rate'] >= TARGET_SWARMAGENTIC

per_query = pd.DataFrame([
    {
        'query_idx': r['query_idx'],
        'final_pass_rate': r.get('evaluation', {}).get('final_pass_rate', 0.0),
        'hard_constraint_macro': r.get('evaluation', {}).get('hard_constraint_macro', 0.0),
        'commonsense_macro': r.get('evaluation', {}).get('commonsense_macro', 0.0),
        'delivery_rate': r.get('evaluation', {}).get('delivery_rate', 0.0),
        'ticks': r.get('total_ticks', 0),
        'tokens_used': r.get('tokens_used', 0),
        'cost_usd': r.get('cost_used', 0.0),
        'stop_reason': r.get('stop_reason', ''),
    }
    for r in results
]).sort_values('query_idx').reset_index(drop=True)

aggregate, per_query


In [ ]:
out_dir = ROOT / 'metrics' / 'output' / 'sprint6_travelplanner'
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
out_file = out_dir / f'benchmark_{stamp}.json'

payload = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'config': {
        'query_indices': QUERY_INDICES,
        'max_ticks': MAX_TICKS,
        'agents': AGENTS,
        'seed': SEED,
        'data_dir': str(DATA_DIR),
        'target_swarmagentic_final_pass_rate': TARGET_SWARMAGENTIC,
    },
    'aggregate': aggregate,
    'runs': results,
}

out_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f'Wrote: {out_file}')
out_file


## Next steps

- Ajuster `QUERY_INDICES` (ex: `list(range(25))`) pour une campagne plus large.
- Si `avg_final_pass_rate < 0.322`, itérer sur prompts/pression/retry bounds puis relancer.
- Conserver les JSON de sortie pour comparaison inter-runs.
